# 3. Stage 1: the slice-quality classifier (ResNet-50)

Order of the notebook: data, split, model, training, learning curves, results.

Two kinds of numbers appear:

- DEMO: one donor-grouped split trained live here (30 to 60 minutes on a 6 GB GPU). Set
  `DEMO_TRAIN = True` to run it. Demo numbers are never quoted.
- DEFINITIVE: the 5-fold donor-grouped cross-validation results, trained by
  `train_quality_cnn.py` and loaded from `models/*/quality_metrics.json`. The
  dissertation quotes only these.


In [1]:
import os, sys, csv, json, glob, collections
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from nb_style import (ROOT, p, TRAIN_COLOR, VAL_COLOR, USABLE_C, UNUSABLE_C,
                      set_seed, plot_confusion, metrics_report, verify, verify_summary)

DEMO_TRAIN  = False       # True -> live single-split training below (needs CUDA, ~30-60 min)
DEMO_EPOCHS = 8
DEMO_SIZE   = 448         # 448px fits a 6 GB card; the definitive runs use 640px

# point the training module at the combined dataset BEFORE importing it, then import the
# exact code the definitive runs used - the notebook reuses it rather than re-implementing
os.environ["QUALITY_CSV"] = p("review_tool", "results", "combined_human_machine_20260818.csv")
os.environ["QUALITY_IMG_DIRS"] = os.pathsep.join(
    [p("review_tool", "webapp", "review_images"), p("data_bulk", "images")])
os.environ["QUALITY_INPUT_SIZE"] = str(DEMO_SIZE)
sys.path.insert(0, ROOT)
import train_quality_cnn as tq

set_seed(tq.SEED)
print(f"training code: train_quality_cnn.py  (seed {tq.SEED}, classes {tq.CLASSES})")

training code: train_quality_cnn.py  (seed 42, classes ['unusable', 'usable'])


## 1. Training data

847 expert labels plus 5,141 machine labels, one row per slide. Where they conflict the
expert wins. Machine rows carry the ensemble's vote fraction as a soft target, so a 4-of-7
slide teaches the model less confidently than a unanimous one.


In [2]:
ids, y, groups, is_human, fracs = tq.load_rows()
n_h = int(is_human.sum())
print(f"slides {len(ids)}   donors {len(set(groups.tolist()))}   "
      f"human {n_h} / machine {len(ids)-n_h}")
print(f"class counts: { {tq.CLASSES[k]: int(v) for k, v in collections.Counter(y.tolist()).items()} }")

fig, ax = plt.subplots(figsize=(8.4, 3.2))
mfr = fracs[~is_human]
ax.hist(mfr, bins=np.linspace(0, 1, 22), color=TRAIN_COLOR, edgecolor=VAL_COLOR)
ax.set_yscale("log")
ax.set_xlabel("soft target = P(usable) from the 7-member vote")
ax.set_ylabel("machine-labelled slides (log)")
ax.set_title("most machine labels are unanimous (0 or 1); the contested middle "
             "self-attenuates during training", fontsize=10)
fig.tight_layout(); plt.show()

slides 5988   donors 43   human 847 / machine 5141
class counts: {'usable': 5127, 'unusable': 861}


## 2. Donor-grouped split

Split by donor, never by slide, so no donor appears on both sides. The assertions below are
the same ones the training script runs.


In [3]:
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=tq.N_FOLDS)
folds = list(gkf.split(np.zeros(len(y)), y, groups))
for k, (tr, te) in enumerate(folds):
    overlap = set(groups[tr].tolist()) & set(groups[te].tolist())
    assert not overlap, f"donor leak in fold {k}: {overlap}"
    te_h = te[is_human[te]]
    assert len(te_h) > 0, f"fold {k}: no human-labelled test rows"
print(f"{tq.N_FOLDS} donor-grouped folds: no donor spans folds  ✓")
print(f"human test rows per fold: {[int(is_human[te].sum()) for _, te in folds]} "
      f"(sum = {sum(int(is_human[te].sum()) for _, te in folds)} = the full expert set)")
print("evaluation contract: models train on everything, but are JUDGED only on the "
      "expert-labelled rows of held-out donors.")

5 donor-grouped folds: no donor spans folds  ✓
human test rows per fold: [148, 220, 162, 142, 175] (sum = 847 = the full expert set)
evaluation contract: models train on everything, but are JUDGED only on the expert-labelled rows of held-out donors.


## 3. Model

ResNet-50 (timm, ImageNet pretrained, fine-tuned end to end), AdamW at 1e-4, mixed
precision, early stopping on validation macro-F1. Class imbalance is handled by class
weighting or a weighted sampler. Blur and brightness augmentation are not used, because
degradation is the thing being classified (Schömig-Markiefka et al., 2021).


In [4]:
print("the exact fold-training entrypoint used by the definitive runs:")
import inspect
print(inspect.getsource(tq.make_crit))
print(inspect.signature(tq.train_one_fold))
print("arm flags (env): QUALITY_TRAIN_HUMAN_ONLY / SOFT_TARGETS / SAMPLER / AUG=geo / TWO_STAGE")

the exact fold-training entrypoint used by the definitive runs:
def make_crit(y_idx, dev):
    """Class-weighted CE (baseline). Under the sampler the classes arrive ~balanced,
    so plain CE is used instead (weighting twice would overshoot)."""
    if USE_SAMPLER:
        return nn.CrossEntropyLoss()
    cnt = collections.Counter(y_idx.tolist())
    w = torch.tensor([len(y_idx) / (len(CLASSES) * cnt.get(c, 1)) for c in range(len(CLASSES))],
                     dtype=torch.float32, device=dev)
    return nn.CrossEntropyLoss(weight=w)

(imgs, y, fracs, is_human, groups, tr, te, dev, epochs)
arm flags (env): QUALITY_TRAIN_HUMAN_ONLY / SOFT_TARGETS / SAMPLER / AUG=geo / TWO_STAGE


## 4. Live training (DEMO)

One split, printed epoch by epoch. Set `DEMO_TRAIN = True` in the first code cell to run
it. The definitive results in section 6 do not depend on this.


In [5]:
hist = None
if DEMO_TRAIN:
    import torch
    dev = torch.device("cuda")
    imgs = tq.preload(ids)
    tr, te = folds[0]
    te_h = te[is_human[te]]
    tr2, va2 = tq.carve_val(tr, y, groups)
    tl = tq.make_loader(imgs, y, fracs, tr2, dev, True)
    vl = tq.make_loader(imgs, y, fracs, va2, dev, False)
    import timm, torch.nn as nn
    model = timm.create_model(tq.ARCH, pretrained=True, num_classes=2).to(dev)
    crit = tq.make_crit(y[tr2], dev)
    opt = torch.optim.AdamW(model.parameters(), lr=tq.LR, weight_decay=tq.WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda")
    from sklearn.metrics import f1_score
    hist = {"train_f1": [], "val_f1": [], "train_loss": []}
    best, best_state, wait = -1, None, 0
    for ep in range(1, DEMO_EPOCHS + 1):
        trl, _, trpd, trgt = tq.run_epoch(model, tl, crit, opt, scaler, dev, True)
        _, _, vpd, vgt = tq.run_epoch(model, vl, crit, opt, scaler, dev, False)
        trf1 = f1_score(trgt, trpd, average="macro"); vf1 = f1_score(vgt, vpd, average="macro")
        hist["train_loss"].append(trl); hist["train_f1"].append(trf1); hist["val_f1"].append(vf1)
        print(f"epoch {ep:2d}  train loss {trl:.3f} F1 {trf1:.3f} | val macro-F1 {vf1:.3f}")
        if vf1 > best:
            best, wait = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= 4: print("early stop"); break
    model.load_state_dict(best_state)
else:
    print("DEMO_TRAIN = False — skipping the live run (definitive results load in §6).")

DEMO_TRAIN = False — skipping the live run (definitive results load in §6).


## 5. Learning curves

Train against validation macro-F1. A gap above about 0.15 at the best epoch means the
model is memorising the training donors.


In [6]:
if hist:
    ep_ax = np.arange(1, len(hist["val_f1"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
    axes[0].plot(ep_ax, hist["train_loss"], color=TRAIN_COLOR, label="train loss")
    axes[0].legend(); axes[0].set_xlabel("epoch")
    axes[1].plot(ep_ax, hist["train_f1"], color=TRAIN_COLOR, label="train macro-F1")
    axes[1].plot(ep_ax, hist["val_f1"], color=VAL_COLOR, label="val macro-F1")
    axes[1].legend(); axes[1].set_xlabel("epoch")
    fig.suptitle("DEMO split — learning curves", fontsize=10); fig.tight_layout(); plt.show()
    b = int(np.argmax(hist["val_f1"]))
    gap = hist["train_f1"][b] - hist["val_f1"][b]
    print(f"train–val gap at best epoch: {gap:+.3f} -> "
          f"{'sizeable gap: some overfitting' if gap > 0.15 else 'small gap: healthy fit'}")
else:
    print("(no live run this session)")

(no live run this session)


## 6. DEFINITIVE results

Every stored 5-fold run in `models/`. The headline model is the 640px run on the 847
expert labels.


In [7]:
runs = []
for f in sorted(glob.glob(p("models", "*", "quality_metrics.json"))):
    d = json.load(open(f, encoding="utf-8"))
    if d.get("smoke"):
        continue
    if "cnn_aggregate" not in d:   # non-CV result files (Arm F single-split, YOLO) are not table rows
        continue
    a = d["cnn_aggregate"]
    runs.append({
        "run": os.path.basename(os.path.dirname(f)),
        "csv": os.path.basename(d["data"]["csv"]),
        "n": d["data"]["n_slides"],
        "px": d["config"]["input_size"],
        "macro_f1": a["macro_f1_mean"], "f1_std": a["macro_f1_std"],
        "auc": a["auc_mean"], "unus_recall": a["recall_unusable_mean"],
        "unus_r_std": a["recall_unusable_std"],
        "_file": f,
    })
print(f"{'run':28s} {'data':38s} {'n':>5s} {'px':>4s} {'macro-F1':>12s} {'AUC':>6s} {'unus.recall':>14s}")
for r in runs:
    print(f"{r['run']:28s} {r['csv']:38s} {r['n']:5d} {r['px']:4d} "
          f"{r['macro_f1']:.3f} ± {r['f1_std']:.3f} {r['auc']:6.3f} "
          f"{r['unus_recall']:8.3f} ± {r['unus_r_std']:.3f}")
head = next(r for r in runs if r["run"] == "quality_garlick640")
d640 = json.load(open(head["_file"], encoding="utf-8"))
cm = np.array(d640["cnn_aggregate"]["confusion_matrix_summed"])
fig, axes = plt.subplots(1, 2, figsize=(10.6, 3.8))
plot_confusion(cm, "headline model: 640px / 847 expert labels\nsummed over 5 donor-held-out folds",
               ax=axes[0])
pf = d640["per_fold"]
axes[1].bar(np.arange(1, 6) - 0.18, [f_["macro_f1"] for f_ in pf], 0.36,
            color=TRAIN_COLOR, label="macro-F1")
axes[1].bar(np.arange(1, 6) + 0.18, [f_["recall"]["unusable"] for f_ in pf], 0.36,
            color=VAL_COLOR, label="unusable recall")
axes[1].axhline(head["macro_f1"], color=TRAIN_COLOR, ls=":", lw=1)
axes[1].set_xlabel("fold (held-out donor group)"); axes[1].legend(fontsize=8)
axes[1].set_title("per-fold spread — why every mean carries a ±", fontsize=9)
fig.tight_layout(); plt.show()
metrics_report("Stage-1 headline (DEFINITIVE, 5-fold donor-grouped CV)",
    **{"model": "ResNet-50, 640px, 847 expert labels",
       "macro-F1": f"{head['macro_f1']:.4f} ± {head['f1_std']:.4f}",
       "AUC-ROC": f"{head['auc']:.4f}",
       "unusable recall": f"{head['unus_recall']:.4f} ± {head['unus_r_std']:.4f}",
       "unusable caught": f"{cm[0,0]}/{cm[0].sum()}",
       "baselines beaten by": f"majority +{d640['beats_majority_by']:.3f}, "
                              f"tissue-fraction +{d640['beats_tissue_fraction_by']:.3f}"})

run                          data                                       n   px     macro-F1    AUC    unus.recall
quality                      final_clean_dataset_20260802.csv         847  448 0.727 ± 0.078  0.856    0.672 ± 0.180
quality_combined_hard        combined_human_machine_20260818.csv     5988  640 0.699 ± 0.034  0.889    0.918 ± 0.029
quality_combined_soft2stage  combined_human_machine_20260818.csv     5988  640 0.758 ± 0.049  0.903    0.836 ± 0.072
quality_garlick640           final_clean_dataset_20260802.csv         847  640 0.762 ± 0.091  0.890    0.738 ± 0.137
quality_union_humanonly      combined_human_machine_20260818.csv     5988  640 0.804 ± 0.021  0.916    0.801 ± 0.044
METRICS FOR REPORT - Stage-1 headline (DEFINITIVE, 5-fold donor-grouped CV)
  model                              ResNet-50, 640px, 847 expert labels
  macro-F1                           0.7623 ± 0.0912
  AUC-ROC                            0.8898
  unusable recall                    0.7377 ± 0.1368
  

## 7. Check against the research log

Aggregates are recomputed from their per-fold entries and compared with `research/LOG.md`.


In [8]:
# combined dataset integrity (already checked in notebook 01; repeated here since
# this notebook trains on it)
verify("combined rows", len(ids), 5988, source="LOG 2026-08-18")
verify("human rows in combined", n_h, 847, source="LOG 2026-08-18")
verify("fractional soft targets", int(((fracs > 0) & (fracs < 1)).sum()), 1186,
       source="LOG 2026-08-18")

# headline model: re-derive the aggregate from the per-fold entries
pf = d640["per_fold"]
verify("headline macro-F1 (re-averaged from folds)",
       round(float(np.mean([f_["macro_f1"] for f_ in pf])), 4), 0.7623,
       source="LOG 2026-08-04")
verify("headline unusable recall (re-averaged)",
       round(float(np.mean([f_["recall"]["unusable"] for f_ in pf])), 4), 0.7377,
       source="LOG 2026-08-04")
cm_re = np.sum([np.array(f_["confusion_matrix"]) for f_ in pf], axis=0)
verify("headline confusion (re-summed from folds)", cm_re.tolist(),
       [[129, 34], [84, 600]], source="LOG 2026-08-04")
verify("headline evaluated on", int(sum(f_["n_test"] for f_ in pf)), 847,
       source="the full expert set, once each")

if DEMO_TRAIN and hist:
    print(f"\nDEMO split best val macro-F1 this run: {max(hist['val_f1']):.3f}")
    print("note: GPU training is seed-fixed but not bit-deterministic — expect the demo "
          "number to vary run-to-run by roughly ±0.02. The definitive numbers above are "
          "stored artefacts and always verify exactly.")
verify_summary()

  ✓  combined rows: computed 5988  expected 5988   [LOG 2026-08-18]
  ✓  human rows in combined: computed 847  expected 847   [LOG 2026-08-18]
  ✓  fractional soft targets: computed 1186  expected 1186   [LOG 2026-08-18]
  ✓  headline macro-F1 (re-averaged from folds): computed 0.7623  expected 0.7623   [LOG 2026-08-04]
  ✓  headline unusable recall (re-averaged): computed 0.7377  expected 0.7377   [LOG 2026-08-04]
  ✓  headline confusion (re-summed from folds): computed [[129, 34], [84, 600]]  expected [[129, 34], [84, 600]]   [LOG 2026-08-04]
  ✓  headline evaluated on: computed 847  expected 847   [the full expert set, once each]

VERIFICATION: 7/7 checks passed


## 8. Independent re-evaluation

Loads the five fold checkpoints of the headline run, rebuilds the same folds, runs
inference on each fold's held-out slides on this machine and recomputes every metric.
Different GPUs can flip slides that sit on the decision boundary, so metrics are checked
to a tolerance of 0.005 macro-F1. Set `RECHECK_DEFINITIVE = False` to skip
(30 to 45 minutes).


In [9]:
RECHECK_DEFINITIVE = True

if RECHECK_DEFINITIVE:
    import torch, timm
    from torchvision import transforms
    dev = torch.device("cuda")
    CKPT_DIR = p("models", "quality_union_humanonly")
    EVAL_PX = 640

    norm = transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    def slide_tensor(img_id):
        with Image.open(tq.find_image(img_id)) as im:
            a = np.asarray(im.convert("RGB").resize((EVAL_PX, EVAL_PX), Image.BILINEAR))
        return norm(torch.from_numpy(a.transpose(2,0,1)).float() / 255.0)

    stored = json.load(open(os.path.join(CKPT_DIR, "quality_metrics.json")))
    cm_re = np.zeros((2,2), int); per_fold_f1 = []
    for k, (tr, te) in enumerate(folds):
        te_h = te[is_human[te]]
        ck = torch.load(os.path.join(CKPT_DIR, f"quality_resnet50_fold{k+1}.pt"),
                        map_location="cpu", weights_only=True)
        assert ck["input_size"] == EVAL_PX and ck["fold"] == k+1
        model = timm.create_model(ck["arch"], pretrained=False, num_classes=2).to(dev).eval()
        model.load_state_dict(ck["state_dict"])
        preds = []
        with torch.no_grad():
            for i in range(0, len(te_h), 4):
                x = torch.stack([slide_tensor(ids[j]) for j in te_h[i:i+4]]).to(dev)
                with torch.autocast("cuda"):
                    preds.append(model(x).float().argmax(1).cpu().numpy())
        preds = np.concatenate(preds); gts = y[te_h]
        from sklearn.metrics import f1_score
        f1k = f1_score(gts, preds, average="macro")
        per_fold_f1.append(f1k)
        for t, pr in zip(gts, preds): cm_re[t, pr] += 1
        st = stored["per_fold"][k]
        print(f"fold {k+1}: recomputed macro-F1 {f1k:.4f}  (stored {st['macro_f1']:.4f})  "
              f"n={len(te_h)}")
        del model; torch.cuda.empty_cache()

    acc_re = (cm_re[0,0]+cm_re[1,1]) / cm_re.sum()
    print(f"\nrecomputed summed confusion: {cm_re.tolist()}")
    print(f"stored   summed confusion: {stored['cnn_aggregate']['confusion_matrix_summed']}")
    agree = 847 - int(np.abs(cm_re - np.array(stored['cnn_aggregate']['confusion_matrix_summed'])).sum()//2)
    verify("re-eval macro-F1 mean (±0.005)", round(float(np.mean(per_fold_f1)),4),
           stored["cnn_aggregate"]["macro_f1_mean"], tol=0.005, source="LOG 2026-08-22")
    verify("re-eval accuracy (±0.005)", round(float(acc_re),4),
           round((np.array(stored['cnn_aggregate']['confusion_matrix_summed']).trace())/847,4),
           tol=0.005, source="LOG 2026-08-22")
    print(f"slide-level agreement with the stored run: >= {agree}/847")
    verify_summary()
else:
    print("RECHECK_DEFINITIVE = False - skipped")

fold 1: recomputed macro-F1 0.7876  (stored 0.7876)  n=148


fold 2: recomputed macro-F1 0.7765  (stored 0.7765)  n=220


fold 3: recomputed macro-F1 0.8364  (stored 0.8341)  n=162


fold 4: recomputed macro-F1 0.8020  (stored 0.8042)  n=142


fold 5: recomputed macro-F1 0.8183  (stored 0.8183)  n=175

recomputed summed confusion: [[132, 31], [83, 601]]
stored   summed confusion: [[132, 31], [83, 601]]
  ✓  re-eval macro-F1 mean (±0.005): computed 0.8041  expected 0.8041   [LOG 2026-08-22]
  ✓  re-eval accuracy (±0.005): computed 0.8654  expected 0.8654   [LOG 2026-08-22]
slide-level agreement with the stored run: >= 847/847

VERIFICATION: 9/9 checks passed


## 9. Full rerun from this notebook (optional, overnight)

Runs the complete 5-fold CV with the same training code and writes the results to
`models/quality_nbrun448/`. This machine trains at 448px, not the 640px used for the
headline run, so numbers will sit slightly lower. Reruns move by about 0.02 because GPU
training is not bit-deterministic. Set `NB_FULL_RUN = True` to start (4 to 6 hours).


In [10]:
NB_FULL_RUN = False

if NB_FULL_RUN:
    import subprocess, sys as _sys
    env = dict(os.environ,
        QUALITY_CSV=p("review_tool","results","combined_human_machine_20260818.csv"),
        QUALITY_IMG_DIRS=os.pathsep.join([p("review_tool","webapp","review_images"),
                                          p("data_bulk","images")]),
        QUALITY_INPUT_SIZE="448", QUALITY_BATCH="12",
        QUALITY_TRAIN_HUMAN_ONLY="1",
        QUALITY_OUT_DIR=p("models","quality_nbrun448"),
        PYTHONUNBUFFERED="1")
    proc = subprocess.Popen([_sys.executable, p("train_quality_cnn.py")],
                            env=env, cwd=ROOT, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    print("\nexit code:", proc.wait())
    print("re-run this notebook afterwards: §6's table will now include quality_nbrun448")
else:
    print("NB_FULL_RUN = False - set True and run for the overnight in-notebook training run")

NB_FULL_RUN = False - set True and run for the overnight in-notebook training run
